In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from sqlalchemy import text

from src.database.load_data import engine

pd.set_option("display.max_rows", 100)

In [2]:
df = pd.read_sql(text("SELECT * FROM feature_table"), engine)
print(df.shape)
df.head()

(307511, 63)


,SK_ID_CURR,TARGET,CODE_GENDER,DAYS_BIRTH,age_years,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,CNT_CHILDREN,CNT_FAM_MEMBERS,...,bureau_balance_dpd_count,bureau_balance_closed_count,goods_price_credit_ratio,income_per_family_member,has_bureau_overdue_flag,has_pos_dpd_flag,has_cc_dpd_flag,total_dpd_count,is_high_utilization,late_payment_rate
0,298488,0,F,-21558,59.1,Secondary / secondary special,Married,House / apartment,0,2.0,...,0.0,185.0,1.0000,18900.0,0,0,0,0,0,0.0000
1,298493,1,M,-16672,45.7,Secondary / secondary special,Married,House / apartment,1,3.0,...,NaN,NaN,0.8347,37500.0,0,0,0,0,0,0.4286
2,298523,0,M,-18745,51.4,Secondary / secondary special,Separated,House / apartment,0,1.0,...,10.0,184.0,0.7953,202500.0,0,1,0,1,0,0.1429
3,298524,0,F,-20547,56.3,Secondary / secondary special,Married,House / apartment,0,2.0,...,5.0,194.0,0.7445,90000.0,0,0,0,0,1,0.0000
4,298530,0,F,-14391,39.4,Higher education,Married,House / apartment,1,3.0,...,0.0,244.0,0.9099,30000.0,0,0,0,0,0,0.0385


In [3]:
## 1. Analisis Missing Value
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_pct
cc_utilization_ratio,221475,72.02
cc_avg_credit_limit,220606,71.74
cc_record_count,220606,71.74
cc_avg_balance,220606,71.74
cc_dpd_count,220606,71.74
bureau_balance_closed_count,215280,70.01
bureau_balance_record_count,215280,70.01
bureau_balance_dpd_count,215280,70.01
EXT_SOURCE_1,173378,56.38
OCCUPATION_TYPE,96391,31.35


## 2. Kategorisasi Kolom & Strategi Imputasi

**Grup A — "Count/Aggregate" dari histori (NULL = tidak punya histori → isi 0)**
Kolom dari bureau, POS, installments, credit card, bureau_balance yang NULL
karena customer memang tidak punya baris terkait di tabel behavioral.

**Grup B — Statistik eksternal (NULL = data hilang → isi median)**
EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3, employment_years

**Grup C — Kategorikal (NULL = tidak dilaporkan → isi 'Unknown')**
OCCUPATION_TYPE

**Grup D — Kolom redundant (drop, karena sudah ada versi turunannya)**
DAYS_BIRTH (sudah ada age_years), DAYS_EMPLOYED (sudah ada employment_years)

In [4]:
# Grup A: fillna(0) - NULL berarti tidak ada histori
group_a_fillzero = [
    "bureau_avg_days_credit", "bureau_avg_credit_sum", "bureau_avg_credit_debt",
    "prev_avg_amt_application", "prev_avg_amt_credit", "prev_avg_cnt_payment",
    "prev_approval_rate",
    "pos_record_count", "pos_avg_installment_count", "pos_avg_dpd",
    "pos_max_dpd", "pos_dpd_count",
    "installment_count", "avg_payment_diff", "avg_days_late",
    "late_payment_count", "late_payment_rate",
    "cc_record_count", "cc_avg_balance", "cc_avg_credit_limit",
    "cc_utilization_ratio", "cc_dpd_count",
    "bureau_balance_record_count", "bureau_balance_dpd_count",
    "bureau_balance_closed_count",
]

# Grup B: impute median - data hilang beneran
group_b_median = [
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "employment_years",
    "annuity_income_ratio", "credit_income_ratio", "goods_price_credit_ratio",
    "income_per_family_member",
    "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_FAM_MEMBERS",   # <-- tambahan baru
]

# Grup C: kategorikal -> isi 'Unknown'
group_c_unknown = ["OCCUPATION_TYPE"]

# Grup D: drop, sudah ada versi turunannya
group_d_drop = ["DAYS_BIRTH", "DAYS_EMPLOYED"]

print("Total kolom terdaftar:", 
      len(group_a_fillzero) + len(group_b_median) + len(group_c_unknown) + len(group_d_drop))

Total kolom terdaftar: 39


In [5]:
# Safety check: pastikan SEMUA kolom yang punya missing value sudah masuk salah satu grup
all_grouped_cols = set(group_a_fillzero + group_b_median + group_c_unknown + group_d_drop)
missing_cols = set(missing_summary[missing_summary["missing_count"] > 0].index)

not_covered = missing_cols - all_grouped_cols
print("Kolom missing yang BELUM masuk grup manapun:", not_covered)

Kolom missing yang BELUM masuk grup manapun: set()


In [6]:
df_processed = df.copy()

# Grup A - fillna 0
for col in group_a_fillzero:
    df_processed[col] = df_processed[col].fillna(0)

# Grup B - fillna median
for col in group_b_median:
    median_val = df_processed[col].median()
    df_processed[col] = df_processed[col].fillna(median_val)

# Grup C - fillna 'Unknown'
for col in group_c_unknown:
    df_processed[col] = df_processed[col].fillna("Unknown")

# Grup D - drop kolom redundant
df_processed = df_processed.drop(columns=group_d_drop)

print("Sisa missing value setelah treatment:")
print(df_processed.isnull().sum().sum())

Sisa missing value setelah treatment:
0


In [7]:
## 3. Encoding Strategy untuk Kolom Kategorikal
categorical_cols = df_processed.select_dtypes(include="object").columns.tolist()
print(categorical_cols)

for col in categorical_cols:
    print(f"\n{col}: {df_processed[col].nunique()} unique values")
    print(df_processed[col].value_counts())

['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']

CODE_GENDER: 3 unique values
CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

NAME_EDUCATION_TYPE: 5 unique values
NAME_EDUCATION_TYPE
Secondary / secondary special    218391
Higher education                  74863
Incomplete higher                 10277
Lower secondary                    3816
Academic degree                     164
Name: count, dtype: int64

NAME_FAMILY_STATUS: 6 unique values
NAME_FAMILY_STATUS
Married                 196432
Single / not married     45444
Civil marriage           29775
Separated                19770
Widow                    16088
Unknown                      2
Name: count, dtype: int64

NAME_HOUSING_TYPE: 6 unique values
NAME_HOUSING_TYPE
House / apartment      272868
With parents            14840
Municipal apartment     11183
Rented apartment         4881
Office a

C:\Users\christovel kevin .m\AppData\Local\Temp\ipykernel_27320\1651408523.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_processed.select_dtypes(include="object").columns.tolist()


## 4. Terapkan Encoding

Strategi:
- `FLAG_OWN_CAR`, `FLAG_OWN_REALTY` (nilai Y/N) -> binary encoding (1/0)
- `CODE_GENDER` -> binary encoding (M/F), baris "XNA" (anomali kecil) di-drop
- Kolom kategorikal lain (education, family status, housing, income type, occupation) -> One-Hot Encoding, karena:
  - Cardinality rendah-sedang (tidak lebih dari ~20 kategori)
  - Tidak ada urutan alami antar kategori (nominal, bukan ordinal)
  - Cocok untuk Logistic Regression (baseline model di Milestone 22)

In [8]:
# Binary encoding
df_processed["FLAG_OWN_CAR"] = df_processed["FLAG_OWN_CAR"].map({"Y": 1, "N": 0})
df_processed["FLAG_OWN_REALTY"] = df_processed["FLAG_OWN_REALTY"].map({"Y": 1, "N": 0})

# Drop anomali CODE_GENDER == 'XNA' (jumlahnya sangat kecil, lihat hasil Milestone 14)
df_processed = df_processed[df_processed["CODE_GENDER"] != "XNA"]
df_processed["CODE_GENDER"] = df_processed["CODE_GENDER"].map({"M": 1, "F": 0})

# One-Hot Encoding untuk sisa kolom kategorikal
onehot_cols = [
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE",
    "NAME_INCOME_TYPE", "OCCUPATION_TYPE"
]

df_processed = pd.get_dummies(df_processed, columns=onehot_cols, drop_first=False)

print(df_processed.shape)
df_processed.head()

(307507, 100)


,SK_ID_CURR,TARGET,CODE_GENDER,age_years,CNT_CHILDREN,CNT_FAM_MEMBERS,AMT_INCOME_TOTAL,employment_years,AMT_CREDIT,AMT_ANNUITY,...,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Unknown,OCCUPATION_TYPE_Waiters/barmen staff
0,298488,0,0,59.1,0,2.0,37800.0,4.5,675000.0,28597.5,...,False,False,False,False,False,False,False,False,True,False
1,298493,1,1,45.7,1,3.0,112500.0,11.9,1078200.0,38331.0,...,False,False,False,False,False,False,False,False,True,False
2,298523,0,1,51.4,0,1.0,202500.0,10.0,605439.0,31041.0,...,False,False,False,False,False,False,False,False,False,False
3,298524,0,0,56.3,0,2.0,180000.0,4.5,803907.0,29002.5,...,False,False,False,False,False,False,False,False,True,False
4,298530,0,0,39.4,1,3.0,90000.0,7.9,247275.0,17716.5,...,False,False,False,False,False,False,False,False,False,False


In [9]:
print("Missing value tersisa:", df_processed.isnull().sum().sum())
print("Total kolom:", df_processed.shape[1])
print("Total baris:", df_processed.shape[0])
print("\nTipe data:")
print(df_processed.dtypes.value_counts())

Missing value tersisa: 0
Total kolom: 100
Total baris: 307507

Tipe data:
bool       44
float64    39
int64      17
Name: count, dtype: int64


In [10]:
## 5. Export Hasil Final
output_path = project_root / "data" / "processed" / "feature_table_final.csv"

df_processed.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Shape: {df_processed.shape}")

Saved to: d:\credit-risk-intelligence-platform - Copy\project\data\processed\feature_table_final.csv
Shape: (307507, 100)


In [11]:
# Convert bool columns ke int dulu (bool sering bikin to_sql lambat)
bool_cols = df_processed.select_dtypes(include="bool").columns.tolist()
df_processed[bool_cols] = df_processed[bool_cols].astype(int)

df_processed.to_sql(
    "feature_table_final",
    engine,
    if_exists="replace",
    index=False,
    chunksize=20000,
    method=None
)

print("feature_table_final berhasil disimpan ke PostgreSQL.")

feature_table_final berhasil disimpan ke PostgreSQL.


In [12]:
# Validasi dari CSV
df_check_csv = pd.read_csv(output_path)
print("CSV shape:", df_check_csv.shape)

# Validasi dari PostgreSQL
df_check_db = pd.read_sql(text("SELECT COUNT(*) FROM feature_table_final"), engine)
print("PostgreSQL row count:", df_check_db.iloc[0, 0])

CSV shape: (307507, 100)
PostgreSQL row count: 307507
